# DeepCGM Tutorial - Knowledge-Guided Deep Learning Crop Growth Model

---

## Learning objectives

After completing this notebook you will be able to:

1. Explain why a pure data-driven LSTM struggles when crop observations are sparse.
2. Understand the role of each gating unit in DeepCGM and connect these concepts to their implementation in code.
3. Describe the knowledge in DeepCGM: **mass conservation** (physical laws), **Physiological process designing**, **Input Mask** (relevant-inputs-only), **Convergence Loss** (stable internal processes).
4. Observe and analyse three experiments:
   - Task 1 - NaiveLSTM on sparse observations;
   - Task 2.1 - DeepCGM with mass conservation structure only;
   - Task 2.2 - DeepCGM with Input Mask and Convergence Loss added (the full configuration).
5. Compare the performance of different model configurations.
6. Feel how models learn/fit the sparse observations.
7. Answer the discussion questions.

## 0.1 Background - why Knowledge-Guided Machine Learning?

### 1. Why do we want machine learning in the first place?

Traditional process-based crop models (ORYZA2000, WOFOST, DSSAT, ...) describe crop growth through equations derived from plant physiology. The catch is that almost every equation in these models is **empirical or semi-empirical**: the functional form (sigmoid, exponential, power law, ...) was chosen by the modeller, and its parameters were calibrated against one specific dataset. Two consequences follow:

- The model inherently encodes **the modeller's biases** — pick a different family of equations, get different predictions for the same field.
- The development cycle is *hypothesis -> experiment -> manual model update*, which is slow and cannot **scale with the modern explosion of agricultural data**.

A data-driven (machine-learning) model is the opposite extreme: feed in the data, let the network find its own equations, with no hand-picked functional form.

### 2. The catch: agricultural observations are sparse

In practice we rarely have continuous measurements of crop state. In this tutorial's dataset, **a 165-day growing season is anchored by only ~5-6 sampling dates** (roughly one observation every 20-30 days). A vanilla LSTM trained on such data:

- has **no fitting target on most days**, so it is free to wiggle wildly between the few observed points;
- memorises the handful of training observations (low *training* RMSE) but generalises poorly to unseen seasons (high *test* RMSE);
- has no built-in guard against unphysical behaviour like negative biomass or a yield curve that drops back to zero after harvest.

Part 5 of this notebook makes this concrete: LSTM achieves the **lowest training RMSE on 5 of 6 variables**, yet its curves are the *least* physical and its *test* RMSE is the *highest*.

### 3. The KGML compromise: borrow knowledge to plug the data gaps

**Knowledge-Guided Machine Learning (KGML)** sits between the two extremes. It keeps the flexibility of a learned model but injects **physical / physiological knowledge** — as architectural constraints, masked inputs, or extra loss terms — to fill in for the missing observations. The knowledge does not have to predict the exact crop state on each unobserved day; it just has to **rule out the implausible behaviours** the data alone cannot eliminate (e.g. "carbon cannot appear or disappear without going through a modelled process").

KGML is a broad idea with applications across hydrology, climate science, materials, and beyond. **In this notebook we focus narrowly on one specific use of KGML: compensating for sparse crop observations** — exactly what DeepCGM (Han et al. 2025) was designed to do.

### Where the three approaches sit

| Approach | Strengths | Limitations |
| --- | --- | --- |
| Process-based models (ORYZA2000, WOFOST) | Interpretable, physiologically grounded | Empirical equations encode the modeller's bias; cannot easily exploit new data |
| Pure data-driven (LSTM) | Flexible, learns directly from data | Black box; data-hungry; over-fits on sparse observations |
| **Knowledge-guided DL (DeepCGM)** | Plausible trajectories even on sparse observations | Higher design cost; requires domain knowledge to bake in |

The core idea of DeepCGM: **let data adapt the model while domain knowledge keeps it physically plausible**.

In the remainder of this notebook, we start from a pure LSTM baseline (Part 2) and add the knowledge layers one block at a time, observing how each one curbs the unphysical behaviour.

## 0.2 Setup and imports - run this first

In [ ]:
#@title Setup: install helper.py and clone upstream DeepCGM
# One-shot setup + all imports for the whole notebook. Run me first.
import os, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import torch

# 1) fetch the tutorial helper once (skipped if already present, so local edits survive)
if not os.path.exists('helper.py'):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/flydephone/DeepCGM_tutorial/main/helper.py", "helper.py")
import helper
print(f"PyTorch {torch.__version__}  |  device: {helper.device}")

---
# Part 1 - Data and pretrained models
## 1.1 Set hyperparameters and load dataset
Loads 105 samples pooled across two years: **65 plot-seasons from 2018** (samples 0-64) and **40 plot-seasons from 2019** (samples 65-104).

In [ ]:
#@title Hyperparameters (SEED, TRA_YEAR, BATCH_SIZE)
SEED       = 0       # which of the 50 robust pretrained runs to load (also seeds the bonus training in Part 6.1)
TRA_YEAR   = "2018"  # which YEAR is the TRAINING set; the other year becomes the test set.
                     #   "2018" -> train on the 65 fields from 2018, test on the 40 fields from 2019
                     #   "2019" -> train on the 40 fields from 2019, test on the 65 fields from 2018
BATCH_SIZE = 128

helper.setup_seed(SEED)

rea_ory, rea_par, rea_wea_fer, rea_spa, rea_int, max_min = helper.load_dataset(TRA_YEAR)
print(f"{len(rea_ory)} plot-seasons available")

## 1.2 Model inputs and outputs

Both LSTM and DeepCGM share **exactly the same daily inputs and outputs** — only the internal architecture between them differs, which makes them directly comparable.

**Daily inputs** (5 features per day, fed at every time step):

| # | Variable | Units | Source | Description |
|---|----------|-------|--------|-------------|
| 1 | Rad     | kJ/m²/day | weather    | Daily solar radiation. Drives potential photosynthesis. |
| 2 | Tmax    | °C        | weather    | Daily maximum temperature. |
| 3 | Tmin    | °C        | weather    | Daily minimum temperature. |
| 4 | N_fert  | kg/ha     | management | Daily nitrogen fertilisation (≈ 2 pulses per season, see plot in 1.3). |
| 5 | DVS     | –         | ORYZA2000  | Crop development stage (0 = sown · 1 = heading · 2 = mature). Provided by the process-based model, **not** learned. |

**Daily outputs** (6 variables per day; the supervised loss only counts the few days where a field observation actually exists):

| # | Variable | Units | Description |
|---|----------|-------|-------------|
| 1 | PAI    | m²/m²     | Plant Area Index. |
| 2 | WLV    | kg/ha     | Leaf biomass. |
| 3 | WST    | kg/ha     | Stem biomass. |
| 4 | WSO    | kg/ha     | Storage-organ biomass. |
| 5 | WAGT   | kg/ha     | Above-ground total biomass (= WLV + WST + WSO). |
| 6 | Yield  | kg/ha     | Final grain yield. |


## 1.3 Look at one growing season

Pick any sample and visualise the meteorology (top row) and the sparse field observations (bottom row).

- Black dots = the sparse, real-world observations the model has to fit.
- Grey lines = the ORYZA2000 process-based simulation.

Notice how few observations there are in a typical growing season - this is exactly why injecting domain knowledge matters.

> ❓ **Discuss with a neighbour**
>
> The grey ORYZA2000 curve was calibrated on the sparse black observation points and it looks "reasonably" close to the observations. Why could a classical crop can be easily calibrated with few data?
>
> What does a process-based model bring to the table that a vanilla LSTM, trained from scratch on the same sparse points, would not have?

In [ ]:
#@title Visualise one growing season (sample #10)
helper.show_sample_overview(sample_idx=10,
                            rea_ory=rea_ory, rea_spa=rea_spa,
                            rea_wea_fer=rea_wea_fer, max_min=max_min)

## 1.4 Train/test split and pretrained models

The split is **cross-year**: with `TRA_YEAR="2018"` the models are **trained on the 65 fields from 2018**, and the 40 fields from 2019 are held out for testing (`TRA_YEAR="2019"` swaps the roles).

We load the three pretrained configurations used below and cache their predictions on **both splits** (training and test): Tasks 1-2.2 look at the training fit, and Part 5 puts training and test side by side to expose over-fitting.

In [ ]:
#@title Split train/test and load the 3 pretrained models
tra_loader, tes_loader, n_tra, n_tes = helper.split_and_load(
    rea_ory, rea_wea_fer, rea_spa, rea_int, tra_year=TRA_YEAR, batch_size=BATCH_SIZE
)
print(f"Train samples: {n_tra:3d}  |  test samples: {n_tes:3d}")

MODELS = ["LSTM", "DeepCGM", "DeepCGM+IM+CG"]   # the three configurations used in this notebook

# predictions on BOTH splits (Tasks use the training set; Part 5 contrasts train vs test)
# verbose=False keeps the cell output clean - we just print one summary at the end.
pretrained_predictions     = helper.load_all_pretrained(tra_loader, max_min, tra_year=TRA_YEAR, seed=SEED, tags=MODELS, verbose=False)
pretrained_predictions_tes = helper.load_all_pretrained(tes_loader, max_min, tra_year=TRA_YEAR, seed=SEED, tags=MODELS, verbose=False)
print(f"Cached predictions on train ({n_tra}) and test ({n_tes}) sets for {len(MODELS)} configurations.")

---
# Part 2 - Task 1: NaiveLSTM on sparse observations

`NaiveLSTM` ([source](https://github.com/WUR-AI/DeepCGM/blob/main/models_aux/NaiveLSTM.py)) is the textbook baseline: a single-layer LSTM with hidden size 64 that maps the daily meteorological inputs to the six target variables. No mass conservation, no physiological structure - just gradient descent on the sparse observations.

Unlike the DeepCGM variants in Parts 3-4 (where we load the authors' 700-epoch pretrained weights), **here you train the NaiveLSTM yourself from scratch**. 100 epochs is enough to reproduce the paper's qualitative behaviour: the model nails the few observed points but wiggles unphysically between them.

> Expected time on Colab CPU: **~30 seconds** for 100 epochs.

In [ ]:
#@title Train your own NaiveLSTM (700 epochs, ~120s on CPU)
# Train a NaiveLSTM from scratch for 700 epochs and time it.
# helper.hands_on_training_lstm prints the configuration, runs train_loop with
# a live progress line, and returns the trained model + the per-epoch loss log.
lstm_model, lstm_log = helper.hands_on_training_lstm(
    tra_loader=tra_loader, tes_loader=tes_loader,
    epochs=700, lr=0.005, seed=SEED,
)

In [ ]:
#@title Plot YOUR trained LSTM on the last training field
# Plot your freshly trained LSTM on the last training field.
pre, spa, ory, _, wea = helper.predict(lstm_model, tra_loader, max_min)
helper.plot_one_sample(pre, spa, ory, wea, sample_loc=-1,
                       title="Task 1: your NaiveLSTM (100 epochs from scratch)")

**What to look for**

- The red model curve passes close to the black observation dots on the days where observations exist.
- But on the unobserved days the curve typically wiggles unrealistically (e.g. Leaf biomass / Stem biomass going up and down before any leaves should exist).

This jagged behaviour on unobserved days is the symptom that motivates DeepCGM.

> ❓ **Discuss with a neighbour**
>
> You just saw that NaiveLSTM nails the observed points but wiggles unphysically between them. Why do you think a vanilla LSTM struggles on a crop-growth task in particular - is it just "too little data", or is there something more fundamental about crops that an unconstrained sequence model cannot capture?

---
# Part 3 - Task 2.1: DeepCGM with mass conservation only

## 3.1 Model structure

DeepCGM combines two layers of domain knowledge:

1. **Mass-conserving structure** — based on MC-LSTM (Hoedt et al., 2021). The cell states represent organ-level carbon pools, and every transfer between them must conserve mass: carbon can move between pools but never appears or disappears out of nowhere.

2. **Physiological process design** — the carbon flow follows an explicit physiological ordering

   `photosynthesis → maintenance respiration → partitioning → growth respiration → redistribution`

   mirroring how a real crop allocates carbon each day.

Both principles are realised inside [`DeepCGM`](https://github.com/WUR-AI/DeepCGM/blob/main/models_aux/DeepCGM_fast.py) as **five physiology-shaped gates**:

| Gate | Shape | Activation | Meaning |
| --- | --- | --- | --- |
| `C_assimilate_gate`     | (1, 1)    | Sigmoid | Photosynthesis: fraction of potential C actually fixed |
| `C_mainResp_gate`       | (1, 24)   | Sigmoid | Maintenance respiration: loss ratio per carbon pool |
| `C_partitation_gate`    | (1, 24)   | Softmax | Partitioning: distributes net C among 24 organ-cells (sums to 1) |
| `C_growResp_gate`       | (1, 24)   | Sigmoid | Growth respiration: per-cell ratio of new growth lost as respiration |
| `C_redistribution_gate` | (24, 24)  | Softmax | Carbon redistribution, leaf->stem->grain etc. (rows sum to 1) |

> The choice of activation reflects the physiology: **Sigmoid** gates produce *independent ratios in [0, 1]* (each pool decides its own loss fraction), while **Softmax** gates produce *distributions that sum to 1* - required for partition and redistribution because mass conservation forces the outflows from one source to sum to the full inflow.

> ❓ **Discuss with a neighbour**
>
> Sigmoid and Softmax are not the only differentiable activations available. Could the authors have used something else and kept the model physically valid?
>
> - For the **Sigmoid gates** (assimilate / mainResp / growResp): could `tanh`, `ReLU`, hard-sigmoid, or a `clamp(0, 1)` replace Sigmoid? Which of them still produces a non-negative ratio in [0, 1], and which would let an organ pool lose more than 100 % of its carbon - or even go negative?
> - For the **Softmax gates** (partition / redistribution): the outputs must be non-negative **and** sum to 1 so that mass conservation holds. Would a normalised ReLU (`ReLU(x) / sum(ReLU(x))`) work? Would Sparsemax? What breaks if the row sums become *less than* 1?

The next two cells - **one with the `rate()` function code, one with the architecture diagrams** - are kept side by side so you can match every line of code to a node in the diagrams.

In [ ]:
# Verbatim from models_aux/DeepCGM.py (with one cosmetic edit: the upstream source
# misspells `partition` as `partitation`; the spelling is restored below for
# readability). Shown here so you can compare it line-by-line with the architecture
# diagrams in the next cell. Defining the function in the notebook namespace is a
# harmless no-op (we never call it).

def rate(self, C_cell, x, C):
    # ---- 1. Gate calculations (one matrix per sub-process) ----------------
    C_assimilate_ratio   = self.C_assimilate_gate    (x, self.C_i_assimilate_prior,     self.C_o_assimilate_prior)
    C_partition_mat      = self.C_partition_gate     (x, self.C_i_partition_prior,      self.C_o_partition_prior)
    C_growResp_mat       = self.C_growResp_gate      (x, self.C_i_growResp_prior,       self.C_o_growResp_prior)
    C_mainResp_mat       = self.C_mainResp_gate      (x, self.C_i_mainResp_prior,       self.C_o_mainResp_prior)
    C_redistribution_mat = self.C_redistribution_gate(x, self.C_i_redistribution_prior, self.C_o_redistribution_prior)

    # ---- 2. Mass-conserving carbon flow (one step) ------------------------
    C_in       = C_assimilate_ratio.squeeze(-2) * C                                    # photosynthesis: how much of C_potential is actually fixed
    C_mainResp = torch.mul(C_cell.unsqueeze(-2), C_mainResp_mat).squeeze(-2)           # maintenance respiration: each pool pays a cost
    C_net      = C_in - C_mainResp.sum(-1, keepdim=True)                               # net assimilation available for growth
    C_grow     = torch.matmul(C_net.unsqueeze(-2), C_partition_mat).squeeze(-2)        # partition net carbon to each organ-cell
    C_growResp = C_grow.unsqueeze(-2) * C_growResp_mat                                 # growth respiration: each new gram costs
    C_grow_net = C_grow.unsqueeze(-2) - C_growResp                                     # net growth per cell
    C_cell     = C_cell + C_grow_net.squeeze(-2)                                       # accumulate into the carbon pools
    C_cell     = torch.matmul(C_cell.unsqueeze(-2), C_redistribution_mat).squeeze(-2)  # redistribute between leaf/stem/storage

    return C_cell, C_assimilate_ratio, C_partition_mat, C_growResp_mat, C_mainResp_mat, C_redistribution_mat, C_net

*Diagram 1 - overall architecture* ([open on GitHub](https://github.com/flydephone/DeepCGM_tutorial/blob/main/figure/DeepCGM_architecture.png)):

![overall](https://raw.githubusercontent.com/flydephone/DeepCGM_tutorial/main/figure/DeepCGM_architecture.png)

> ❓ **Discuss with a neighbour**
>
> Trace through the diagram and verify that the model processes carbon flow in this exact order:
>
> `photosynthesis → maintenance respiration → partitioning → growth respiration → redistribution`
>
> Can you identify each of the five steps in the figure, and match each one to the corresponding line of the `rate()` code above?

*Diagram 2 - detailed sub-processes* ([open on GitHub](https://github.com/WUR-AI/DeepCGM/blob/main/figure/DeepCGM_detail.svg)):

![detail](https://raw.githubusercontent.com/WUR-AI/DeepCGM/main/figure/DeepCGM_detail.svg)

> ❓ **Discuss with a neighbour**
>
> Look at the **shape of each of the five gates** in the diagram and trace **how each gate controls the carbon flow**:
>
> - What is each gate's output shape? Which are scalar (1×1), which are row vectors (1×24), which is a full matrix (24×24)?
> - How does that shape determine the way the gate routes carbon - a single ratio, a per-pool ratio, a distribution over 24 pools, or a 24×24 transition matrix?
> - Why would partitioning need a row that sums to 1, but maintenance respiration not?

## 3.2 First feel the wait, then look at the pretrained DeepCGM

DeepCGM is **slower per epoch** than NaiveLSTM (~5x on CPU) because every step has to flow through the five physiology-shaped gates and a 24-cell carbon pool. Before we look at the actual pretrained result, train DeepCGM yourself for **just 20 epochs** so you can measure the wall-time on your own runtime - and then imagine paying that 35x more to reach the paper's 700 epochs.

In [ ]:
#@title Train DeepCGM+IM+CG for 20 epochs (feel the wait)
# Train DeepCGM+IM+CG for 20 epochs to feel the wait.
# We do NOT use the resulting model for the qualitative result below - 20 epochs
# is far too short to converge. The real Task 2.1 result in the next cell still
# comes from the 700-epoch pretrained weights.
hands_on_model, hands_on_log = helper.hands_on_training(
    tra_loader=tra_loader, tes_loader=tes_loader,
    epochs=20, lr=0.1, seed=SEED,
)

**Now look at the 700-epoch pretrained version** - this is what ~35x more training would give you:

*Shown: the model fit on the **training** set (the fields the models were trained on); the panel is the last training field.*

In [ ]:
#@title Display the pretrained DeepCGM (Task 2.1)
helper.show_task_result("DeepCGM", pretrained_predictions,
                        title="Task 2.1: DeepCGM (base, no IM / no CG, 700-epoch pretrained)")

**Observation** - compared to the LSTM red line in Part 2, the DeepCGM red curve is already much more *crop-like*. Because mass conservation is built into the architecture, **biomass can no longer appear or disappear without a reason**: look at **Above-ground biomass** in the early season - it only ever grows gradually, instead of jumping up and down. The curve does, however, still show **violent fluctuations**. These come from the **partition gate**: on the many days with no observation to anchor it, the sparse data let it over-fit, and that over-fitting shows up as the abrupt mid-season swings. Removing exactly these swings is what the Input Mask and Convergence Loss in Task 2.2 are designed for.

---
# Part 4 - Task 2.2: DeepCGM + Input Mask + Convergence Loss (full configuration)

This is the configuration the paper calls "DeepCGM" without qualifiers - mass conservation plus two extra knowledge layers added on top. The training flowchart below ([source](https://github.com/WUR-AI/DeepCGM/blob/main/figure/Training.svg)) shows how the **fitting loss**, the **Input Mask** and the **Convergence Loss** work together during training:

![Training flowchart](https://raw.githubusercontent.com/WUR-AI/DeepCGM/main/figure/Training.svg)

## 4.1 The two new constraints

### Input Mask (relevant inputs only)

Each sub-process should only see the inputs it physically depends on. With the Input Mask on, the inter-organ carbon redistribution depends only on each organ's current carbon stock and the developmental stage (DVS) - **not** on today's weather (Rad / Tmax / Tmin) or fertiliser. This is the *relevant-inputs-only* principle.

### Convergence Loss (stable internal processes)

On top of the fitting loss on the sparse observations, training adds a convergence loss: the cell state from the normal forward iteration is compared with an independently redistributed state at every time step, and their difference is penalised. This keeps adjacent time steps from jumping unrealistically when no observation is anchoring them - the *stable internal processes* principle.

## 4.2 Results - the full DeepCGM (IM + CG)

*Shown: the model fit on the **training** set (the fields the models were trained on); the panel is the last training field.*

In [ ]:
#@title Display the pretrained DeepCGM+IM+CG (Task 2.2)
helper.show_task_result("DeepCGM+IM+CG", pretrained_predictions,
                        title="Task 2.2: DeepCGM + IM + CG (full, 700-epoch pretrained)")

**Observation** - this is the configuration the paper reports. The curves are smooth, monotonic where physiology requires it, and match both ORYZA and the sparse observations even on days with no anchor point.

---
# Part 5 - Side-by-side comparison: training vs test

A single grid for the three models: the **left three columns are the training set** (a field the models were trained on), the **right three columns are the test set** (an unseen field from the held-out year), separated by a blank column. Rows are variables.

The RMSE table below lists train and test next to each other - watch the ranking **flip**.

In [ ]:
#@title Side-by-side training vs test grid (Part 5)
helper.compare_train_test(pretrained_predictions, pretrained_predictions_tes, tags=MODELS)

In [ ]:
#@title Train vs test RMSE table
df_rmse = helper.rmse_train_test_table(pretrained_predictions, pretrained_predictions_tes, tags=MODELS)

**Conclusions - the over-fitting story:**

1. **On the training set, LSTM has the *lowest* RMSE for 5 of 6 variables** - the unconstrained black box bends hardest to hit the sparse points it was trained on, but its curves oscillate unphysically between them. (Yield is the only exception, where DeepCGM is already the best on the training set.)
2. **On the test set, LSTM has the *highest* RMSE for the same 5 variables** - that tight training fit was over-fitting; it does not transfer to the unseen held-out year. (Yield again is the exception - all three models land within ~3% of each other on test Yield.)
3. **DeepCGM and DeepCGM+IM+CG move the opposite way**: slightly *worse* on train (the knowledge constraints stop them chasing every point), but *better* on test, because the physically plausible trajectories generalise.
4. This flip - LSTM best on train / worst on test, the knowledge-guided models the reverse - is the core argument for knowledge-guided learning on sparse data. The gap is largest for **Leaf biomass, Stem biomass, Storage-organ biomass and Above-ground biomass** (test RMSE roughly halved going from LSTM to DeepCGM+IM+CG), the variables with the sparsest mid-season observations.

---
# Part 6 - Bonus: pre-rendered 700-epoch training evolution

The released `model_weight/` directory only ships the final-state checkpoint of each robust run (typically epoch 670-700), so the **full** epoch-by-epoch trajectory is not on disk. The cell below loads a **pre-rendered** 72-frame GIF covering epochs 0 -> 700 in steps of 10 (LSTM vs DeepCGM+IM+CG side by side), so you can see the full convergence story without paying the 15-minute compute cost again.

> The GIF was generated once with `helper.make_evolution_gif(lstm_snaps, dcgm_snaps, ...)` and stored in this tutorial repo. You already felt how slow training is in Section 2 (100-epoch LSTM, ~30 s) and Section 3.2 (20-epoch DeepCGM, ~25 s).

In [ ]:
#@title Download + show the pre-rendered 700-epoch GIF
# Download (once) and display the pre-rendered 700-epoch training-evolution GIF.
helper.show_evolution_gif()

**What to watch in the 700-epoch animation**

- **Early (epoch 0-50)**: both rows are still mostly noise.
- **Middle (epoch 50-200)**: LSTM has already pinned the observation points but keeps oscillating between them; DeepCGM gradually grows physically plausible S-curves.
- **Late (epoch 200-700)**: LSTM's oscillations barely improve - more epochs cannot fix the lack of physical constraints. DeepCGM keeps smoothing and tracks ORYZA closely.


---
# Part 7 - Discussion questions (must-do)

Answer in short paragraphs - no additional experiments are needed.

1. **Fit vs plausibility.** In Part 5 the LSTM has the *lowest* training RMSE for 5 of 6 variables, yet its curves are the least physical and its *test* RMSE is the highest. In your own words, what do mass conservation, the Input Mask and the Convergence Loss buy us in exchange for *raising* the training loss? Frame your answer in terms of generalising to unseen growing seasons rather than fitting the points the model was trained on.

2. **Observation density.** This dataset has ~5-6 observations per growing season. Imagine the opposite extreme: a measurement for every variable on every day. How would the relative benefit of the three knowledge layers (mass conservation / Input Mask / Convergence Loss) change? Rank them from "still essential" to "now nice-to-have", and justify your ranking.

3. **Limitations of DeepCGM.** The list below summarises some potential limitations. For each one, think about:

   1. **Do you agree it is a real problem?** How serious is it for an actual agronomic use case (yield forecasting, fertiliser planning, variety screening, ...)?
   2. **Is there a plausible solution?** Propose how you would address it - architectural, data, training strategy, or otherwise.
   3. **Anything missing?** Are there limitations that you would add to this list?

   The limitations:

   - **Scope is limited to basic biomass.** DeepCGM only simulates biomass and PAI; water availability, soil texture, cultivar differences and pest / disease are out of scope, so situations where these dominate are not covered.
   - **Mass conservation excludes non-conserving variables.** The MC-LSTM backbone cannot represent variables that violate mass conservation, the most obvious being **DVS (phenology)** — that is why DVS is read in from ORYZA2000 rather than learned.
   - **No transfer across location or cultivar.** The 700-epoch weights were trained on one rice cultivar; the current architecture provides no mechanism to adapt to a different site or variety.
   - **Knowledge integration is still manual and incomplete.** Mass conservation, Input Mask and Convergence Loss were all hand-designed, while existing crop models (ORYZA, WOFOST, DSSAT, ...) already encode decades of domain knowledge that the network does not yet use.

---
## References

- Han et al. (2025). *Knowledge-guided machine learning with multivariate sparse data for crop growth modelling*. Field Crops Research. [DOI 10.1016/j.fcr.2025.109885](https://doi.org/10.1016/j.fcr.2025.109885)
- Hoedt et al. (2021). *MC-LSTM: Mass-Conserving LSTM*. ICML.
- Upstream code: https://github.com/WUR-AI/DeepCGM
- This tutorial: https://github.com/flydephone/DeepCGM_tutorial